In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
BATCH_SIZE = 128
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3


In [14]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616],
    ),
])


In [ ]:
train_dataset = CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

test_dataset = CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
)

classes = train_dataset.classes
num_classes = len(classes)

print("Device:", DEVICE)
print("Classes:", classes)


  0%|          | 0/170498071 [00:00<?, ?it/s]

In [ ]:

class Vectorizer(nn.Module):
    """
    Превращает изображение 3x32x32 в вектор признаков размерности 256.
    """
    def __init__(self, embedding_dim=256):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )

        self.flatten = nn.Flatten()
        self.projection = nn.Linear(128, embedding_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.projection(x)
        return x


In [ ]:
class Classifier(nn.Module):
    """
    Классифицирует вектор признаков.
    """
    def __init__(self, embedding_dim=256, num_classes=10):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, embedding_dim=256, num_classes=10):
        super().__init__()

        self.vectorizer = Vectorizer(embedding_dim)
        self.classifier = Classifier(embedding_dim, num_classes)

    def forward(self, x):
        embedding = self.vectorizer(x)
        logits = self.classifier(embedding)
        return logits, embedding

In [ ]:
model = ImageClassifier(
    embedding_dim=256,
    num_classes=num_classes,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [ ]:
def train_one_epoch():
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in train_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()

        logits, _ = model(images)
        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)

    return total_loss / total, correct / total

In [ ]:
@torch.no_grad()
def evaluate():
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in test_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        logits, _ = model(images)
        loss = criterion(logits, targets)

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)

    return total_loss / total, correct / total


for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch()
    test_loss, test_acc = evaluate()

    print(
        f"Epoch {epoch:02d} | "
        f"train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
        f"test loss: {test_loss:.4f}, acc: {test_acc:.4f}"
    )

torch.save(model.state_dict(), "cifar10_vector_classifier.pth")
print("Saved: cifar10_vector_classifier.pth")

In [ ]:
# ResNet


In [ ]:
import torch
from PIL import Image
from torchvision import models, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

# Предобученная модель
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model = model.to(device).eval()

# Убираем финальный классификатор: остаётся feature extractor
model.fc = torch.nn.Identity()

# Такая же предобработка, как при обучении на ImageNet
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

@torch.inference_mode()
def embed(image_path: str) -> torch.Tensor:
    img = Image.open(image_path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(device)  # [1, 3, 224, 224]
    vec = model(x)                                # [1, 2048] для ResNet-50
    return torch.nn.functional.normalize(vec, dim=-1).squeeze(0)

v1 = embed("photo1.jpg")
v2 = embed("photo2.jpg")

# Сходство изображений
similarity = (v1 @ v2).item()
print(similarity)  # от -1 до 1; ближе к 1 — более похожи